# Solar Threads
## A small, honest instance-segmentation baseline

![A small sun researcher points to dark filaments on a warm solar disk.](https://raw.githubusercontent.com/cubres/filament-segmentation-2026/main/assets/solar-filament-hero.png)

*Generated decorative illustration. This is not a solar observation.*

Find each dark filament in a solar image. Give each filament its own mask.
This notebook uses the competition images and training masks. It uses no extra labels or weights.

This is a fast CPU baseline. It is not a claim of a top score.


### 1. Follow one clear path
Fit thresholds on training images. Keep separate months for validation.
Use the selected method on every test image. Check the CSV before a submission.

In [ ]:
import importlib.util
import subprocess
import sys
if importlib.util.find_spec("pycocotools") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "pycocotools==2.0.11"], check=True)
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from IPython.display import display

plt.rcParams.update({"figure.facecolor": "#fffaf2", "axes.facecolor": "#fffaf2",
                     "font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

def flow_diagram(labels, caption):
    fig, ax = plt.subplots(figsize=(14, 2.3))
    ax.set(xlim=(0, len(labels)), ylim=(0, 1)); ax.axis("off")
    colors = ["#ffe3b0", "#cde8dc", "#e6dcf5", "#ffd5cb", "#d7e8f4"]
    for i, label in enumerate(labels):
        ax.add_patch(FancyBboxPatch((i + .04, .28), .84, .48, boxstyle="round,pad=0.03", fc=colors[i % len(colors)], ec="#554b46"))
        ax.text(i + .46, .52, label, ha="center", va="center", fontsize=11)
        if i < len(labels) - 1:
            ax.annotate("", (i + 1.02, .52), (i + .90, .52), arrowprops={"arrowstyle": "->", "lw": 1.6})
    fig.suptitle(caption, fontsize=14); plt.show()

flow_diagram(["Official\nimages + masks", "Separate\ncalendar months", "Fit dark-core\nthresholds", "Measure\nheld-out PQ", "All test images\n→ checked CSV"], "One reproducible path")

### 2. Remove noise. Keep dark structures.
A small blur reduces image grain. A wider blur estimates local brightness.
The contrast map measures how dark each pixel is next to its surroundings.

Our test idea is **dark-core growth**. Keep a faint region only if it contains a dark core.
This can recover edges and reject weak noise. It can also join nearby filaments.
We measure that trade-off. The fixed-threshold method remains the control.

In [ ]:
"""Original CPU solar-filament baseline. Competition masks are the only labels.

MIT license; copyright (c) 2026 prvsiyan / cubres.
The evaluator follows the public host reference. It is not Kaggle's private scorer.
"""
from __future__ import annotations

from collections import defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
import hashlib
import importlib.metadata
import json
import time

import cv2
import numpy as np
import pandas as pd
from pycocotools import mask as coco_mask


@dataclass(frozen=True)
class Config:
    name: str = "fixed_threshold"
    low: float = 0.075
    high: float = 0.075
    blur_sigma: float = 18.0
    denoise_sigma: float = 1.6
    min_area: int = 200
    min_core_area: int = 30
    border: int = 24
    closing_size: int = 3


BASELINE = Config()
CANDIDATES = [
    BASELINE,
    Config("core_growth_050_100", low=0.050, high=0.100),
    Config("core_growth_060_110", low=0.060, high=0.110),
    Config("core_growth_070_120", low=0.070, high=0.120),
    Config("core_growth_050_130", low=0.050, high=0.130),
]


def encode_mask(mask: np.ndarray) -> dict:
    return coco_mask.encode(np.asfortranarray(mask.astype(np.uint8)))


def read_image(path: Path) -> np.ndarray:
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None or image.shape != (2048, 2048):
        raise ValueError(f"Expected one 2048 by 2048 image: {path.name}")
    return image


def image_features(image: np.ndarray, config: Config = BASELINE) -> tuple[np.ndarray, np.ndarray]:
    """Measure dark local contrast. Erode the largest bright disk at its edge."""
    if image.ndim != 2 or not np.isfinite(image).all():
        raise ValueError("Input must be a finite grayscale image")
    bright = (image > 30).astype(np.uint8)
    count, labels, stats, _ = cv2.connectedComponentsWithStats(bright, connectivity=8)
    disk = np.zeros(image.shape, np.uint8)
    if count > 1:
        disk[labels == 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))] = 1
    # Fill dark holes in the solar disk before the edge is eroded.
    contours, _ = cv2.findContours(disk, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        cv2.drawContours(disk, contours, -1, 1, thickness=cv2.FILLED)
    radius = max(1, config.border)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * radius + 1, 2 * radius + 1))
    disk = cv2.erode(disk, kernel).astype(bool)
    smooth = cv2.GaussianBlur(image.astype(np.float32), (0, 0), config.blur_sigma)
    denoised = cv2.GaussianBlur(image.astype(np.float32), (0, 0), config.denoise_sigma)
    contrast = (smooth - denoised) / np.maximum(smooth, 24.0)
    contrast[~disk] = 0
    return contrast, disk


def predict_features(contrast: np.ndarray, disk: np.ndarray, config: Config) -> np.ndarray:
    """Keep a weak dark component only when it contains a strong dark core.

    This seed test is our ablation. Output instances are disjoint by construction.
    The method can merge nearby filaments. That error is visible in PQ.
    """
    weak = ((contrast >= config.low) & disk).astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (config.closing_size, config.closing_size))
    weak = cv2.morphologyEx(weak, cv2.MORPH_CLOSE, kernel)
    weak[~disk] = 0
    count, labels, stats, _ = cv2.connectedComponentsWithStats(weak, connectivity=8)
    strong = (contrast >= config.high) & disk
    core_counts = np.bincount(labels[strong], minlength=count)
    keep = (stats[:, cv2.CC_STAT_AREA] >= config.min_area) & (core_counts >= config.min_core_area)
    keep[0] = False
    mapping = np.zeros(count, np.int32)
    mapping[np.flatnonzero(keep)] = np.arange(1, int(keep.sum()) + 1)
    return mapping[labels]


def predict_image(image: np.ndarray, config: Config) -> np.ndarray:
    return predict_features(*image_features(image, config), config)


def labels_to_rles(labels: np.ndarray) -> list[dict]:
    """Encode sparse label positions once; avoid one full-image scan per instance."""
    height, width = labels.shape
    flat = labels.ravel(order="F")
    positions = np.flatnonzero(flat)
    if not len(positions):
        return []
    order = np.argsort(flat[positions], kind="stable")
    positions = positions[order]
    values = flat[positions]
    splits = np.flatnonzero(np.diff(values)) + 1
    rles = []
    for pixels in np.split(positions, splits):
        starts = np.r_[0, np.flatnonzero(np.diff(pixels) != 1) + 1]
        ends = np.r_[starts[1:] - 1, len(pixels) - 1]
        run_starts, run_ends = pixels[starts], pixels[ends] + 1
        gaps = run_starts - np.r_[0, run_ends[:-1]]
        lengths = run_ends - run_starts
        counts = np.column_stack([gaps, lengths]).ravel().tolist()
        if run_ends[-1] < flat.size:
            counts.append(int(flat.size - run_ends[-1]))
        rles.append(coco_mask.frPyObjects({"size": [height, width], "counts": counts}, height, width))
    return rles


def annotation_rle(annotation: dict, height: int = 2048, width: int = 2048) -> dict:
    # Deliberately use segmentation only: no spines, boxes, or class labels.
    segmentation = annotation["segmentation"]
    if isinstance(segmentation, list):
        return coco_mask.merge(coco_mask.frPyObjects(segmentation, height, width))
    if isinstance(segmentation["counts"], list):
        return coco_mask.frPyObjects(segmentation, height, width)
    rle = dict(segmentation)
    if isinstance(rle["counts"], str):
        rle["counts"] = rle["counts"].encode("ascii")
    return rle


def pq_counts(ground_truth: list[dict], predictions: list[dict]) -> dict:
    """Host-reference semantics: all pairs above IoU 0.5; no Hungarian rewrite."""
    ng, npred = len(ground_truth), len(predictions)
    if ng == 0 or npred == 0:
        return {"iou_sum": 0.0, "tp": 0, "fp": npred, "fn": ng}
    iou = coco_mask.iou(predictions, ground_truth, [0] * ng).T
    hits = iou > 0.5
    return {
        "iou_sum": float(iou[hits].sum()),
        "tp": int(hits.sum()),
        "fp": int((~hits.any(axis=0)).sum()),
        "fn": int((~hits.any(axis=1)).sum()),
    }


def aggregate_pq(rows: list[dict]) -> float:
    denominator = sum(row["tp"] + 0.5 * row["fp"] + 0.5 * row["fn"] for row in rows)
    return sum(row["iou_sum"] for row in rows) / denominator if denominator else 0.0


def stable_key(text: str) -> str:
    return hashlib.sha256(("solar-threads-v1:" + text).encode()).hexdigest()


def split_names(names: list[str], calibration_limit: int = 24, validation_limit: int = 16) -> tuple[list[str], list[str]]:
    """A fixed calendar-month split keeps same-date and annotator copies together."""
    unique = sorted(set(names), key=stable_key)
    calibration = [name for name in unique if int(stable_key(name[:6])[:8], 16) % 4 != 0]
    validation = [name for name in unique if int(stable_key(name[:6])[:8], 16) % 4 == 0]
    if not calibration or not validation:
        raise ValueError("Both month groups need images. Do not improvise a split after seeing scores.")
    calibration, validation = calibration[:calibration_limit], validation[:validation_limit]
    assert not {n[:6] for n in calibration} & {n[:6] for n in validation}
    return calibration, validation


def load_annotations(path: Path) -> tuple[dict, dict]:
    data = json.loads(path.read_text())
    by_name = defaultdict(list)
    by_id = defaultdict(list)
    for item in data["images"]:
        by_name[Path(item["file_name"]).name].append(item["id"])
    for annotation in data["annotations"]:
        # Restrict retained fields to the allowed mask task.
        by_id[annotation["image_id"]].append({"segmentation": annotation["segmentation"]})
    return dict(by_name), dict(by_id)


def evaluate(paths: dict[str, Path], names: list[str], by_name: dict, by_id: dict,
             configs: list[Config]) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    for index, name in enumerate(names):
        image = read_image(paths[name])
        features = image_features(image)
        groups = {image_id: [annotation_rle(a) for a in by_id[image_id]] for image_id in by_name[name]}
        for config in configs:
            start = time.perf_counter()
            labels = predict_features(*features, config)
            predictions = labels_to_rles(labels)
            seconds = time.perf_counter() - start
            for image_id, truth in groups.items():
                rows.append({"config": config.name, "image": name, "annotator_image": image_id,
                             "prediction_count": len(predictions), "seconds": seconds,
                             **pq_counts(truth, predictions)})
        if (index + 1) % 4 == 0 or index + 1 == len(names):
            print(f"Measured {index + 1}/{len(names)} images", flush=True)
    detail = pd.DataFrame(rows)
    summary = pd.DataFrame([
        {"config": config.name, "pq": aggregate_pq(detail[detail.config == config.name].to_dict("records")),
         "images": len(names), "annotator_records": int((detail.config == config.name).sum())}
        for config in configs
    ])
    return summary, detail


def discover(root: Path) -> tuple[Path, dict[str, Path], list[Path]]:
    annotations = list(root.rglob("*Annotations*train*.json"))
    if len(annotations) != 1:
        raise ValueError(f"Expected one official training annotation file, found {len(annotations)}")
    train = {p.name: p for p in root.rglob("*.jpeg") if "train_images" in p.parts}
    test = sorted(p for p in root.rglob("*.jpeg") if "test_images" in p.parts)
    if not train or not test:
        raise ValueError("Attach the official competition data. Train and test images are required.")
    if len(test) != 180:
        raise ValueError(f"Expected the current 180-image test release, found {len(test)}; review data drift.")
    return annotations[0], train, test


def validate_submission(frame: pd.DataFrame, test_stems: set[str], decode_all: bool = True) -> dict:
    if list(frame.columns) != ["filament_id", "segmentation_rle"]:
        raise ValueError("Submission columns changed")
    if frame.empty or frame.filament_id.duplicated().any() or frame.isna().any().any():
        raise ValueError("Empty, duplicate, or missing submission values")
    seen = set()
    rows = 0
    for stem, group in frame.groupby(frame.filament_id.str.rsplit("_", n=1).str[0]):
        if stem not in test_stems:
            raise ValueError(f"Unknown test image: {stem}")
        seen.add(stem)
        coverage = np.zeros((2048, 2048), dtype=bool)
        for counts in group.segmentation_rle:
            if not isinstance(counts, str) or not counts or "\n" in counts:
                raise ValueError("Invalid COCO RLE string")
            rle = {"size": [2048, 2048], "counts": counts.encode("ascii")}
            if decode_all:
                mask = coco_mask.decode(rle).astype(bool)
                if not mask.any() or (coverage & mask).any():
                    raise ValueError("Empty or overlapping predicted masks")
                coverage |= mask
            rows += 1
    # A valid image may have no predicted filaments. Record it, never invent a mask.
    return {"rows": rows, "represented_images": len(seen),
            "zero_prediction_images": sorted(test_stems - seen), "decoded_all": decode_all}


def infer(test_paths: list[Path], config: Config, output_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows, audit = [], []
    for index, path in enumerate(test_paths):
        started = time.perf_counter()
        labels = predict_image(read_image(path), config)
        rles = labels_to_rles(labels)
        for instance, rle in enumerate(rles, 1):
            rows.append({"filament_id": f"{path.stem}_{instance}",
                         "segmentation_rle": rle["counts"].decode("ascii")})
        audit.append({"image_id": path.stem, "filaments": len(rles),
                      "seconds": time.perf_counter() - started,
                      "input_sha256": hashlib.sha256(path.read_bytes()).hexdigest()})
        if (index + 1) % 20 == 0 or index + 1 == len(test_paths):
            print(f"Predicted {index + 1}/{len(test_paths)} test images", flush=True)
    frame = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])
    checks = validate_submission(frame, {p.stem for p in test_paths})
    output_dir.mkdir(parents=True, exist_ok=True)
    frame.to_csv(output_dir / "submission.csv", index=False)
    pd.DataFrame(audit).to_csv(output_dir / "inference-audit.csv", index=False)
    (output_dir / "submission-checks.json").write_text(json.dumps(checks, indent=2))
    return frame, pd.DataFrame(audit)


def run(root: Path, output_dir: Path) -> dict:
    started = time.perf_counter()
    output_dir.mkdir(parents=True, exist_ok=True)
    annotation_path, train_paths, test_paths = discover(root)
    by_name, by_id = load_annotations(annotation_path)
    available = sorted(set(train_paths) & set(by_name))
    calibration, validation = split_names(available)
    print(f"{len(available)} train images. {len(test_paths)} test images.")
    print(f"Fit thresholds on {len(calibration)} images. Hold out {len(validation)} images.")
    cal_summary, cal_detail = evaluate(train_paths, calibration, by_name, by_id, CANDIDATES)
    best_name = cal_summary.sort_values(["pq", "config"], ascending=[False, True]).iloc[0].config
    selected = next(config for config in CANDIDATES if config.name == best_name)
    validation_configs = [BASELINE] if selected == BASELINE else [BASELINE, selected]
    val_summary, val_detail = evaluate(train_paths, validation, by_name, by_id, validation_configs)
    for filename, table in [("calibration-summary.csv", cal_summary), ("calibration-detail.csv", cal_detail),
                            ("validation-summary.csv", val_summary), ("validation-detail.csv", val_detail)]:
        table.to_csv(output_dir / filename, index=False)
    manifest = {"calibration": calibration, "validation": validation,
                "split_unit": "calendar_month", "selected_without_validation": asdict(selected)}
    (output_dir / "split-manifest.json").write_text(json.dumps(manifest, indent=2))
    print("Selected from calibration only:", selected.name)
    print(val_summary.to_string(index=False))
    submission, timing = infer(test_paths, selected, output_dir)
    result = {"kind": "local_holdout_and_submission_artifact", "official_score": None,
              "selected_config": asdict(selected), "local_validation": val_summary.to_dict("records"),
              "train_images_available": len(available), "test_images": len(test_paths),
              "submission_rows": len(submission), "runtime_seconds": time.perf_counter() - started,
              "annotation_sha256": hashlib.sha256(annotation_path.read_bytes()).hexdigest(),
              "submission_sha256": hashlib.sha256((output_dir / "submission.csv").read_bytes()).hexdigest(),
              "packages": {name: importlib.metadata.version(name) for name in
                           ["numpy", "pandas", "pycocotools", "matplotlib"]}}
    (output_dir / "run-report.json").write_text(json.dumps(result, indent=2))
    return result

In [ ]:
cv2.setNumThreads(2)
INPUT = Path("/kaggle/input")
OUTPUT = Path("/kaggle/working")
annotation_path, train_paths, test_paths = discover(INPUT)
by_name, by_id = load_annotations(annotation_path)
available = sorted(set(train_paths) & set(by_name))
calibration, validation = split_names(available)
started = time.perf_counter()

display(pd.DataFrame([
    {"item": "Physical training images", "count": len(available)},
    {"item": "Training annotator records", "count": sum(len(by_name[n]) for n in available)},
    {"item": "Images used to fit thresholds", "count": len(calibration)},
    {"item": "Held-out images", "count": len(validation)},
    {"item": "Test images", "count": len(test_paths)},
]))
flow_diagram(["Local brightness", "Dark contrast", "Weak region +\nstrong core", "Separate\ncomponents", "COCO masks"], "The image method")

### 3. Keep validation separate
Some images have several annotators. All copies of an image stay together.
All images from one calendar month stay together too.

Fit five small settings on 24 images. Select one setting from these results only.
Then measure it on 16 held-out images. This small check has sampling error.

In [ ]:
cal_summary, cal_detail = evaluate(train_paths, calibration, by_name, by_id, CANDIDATES)
best_name = cal_summary.sort_values(["pq", "config"], ascending=[False, True]).iloc[0].config
selected = next(config for config in CANDIDATES if config.name == best_name)
validation_configs = [BASELINE] if selected == BASELINE else [BASELINE, selected]
val_summary, val_detail = evaluate(train_paths, validation, by_name, by_id, validation_configs)
for filename, table in [("calibration-summary.csv", cal_summary), ("calibration-detail.csv", cal_detail),
                        ("validation-summary.csv", val_summary), ("validation-detail.csv", val_detail)]:
    table.to_csv(OUTPUT / filename, index=False)
manifest = {"calibration": calibration, "validation": validation, "split_unit": "calendar_month",
            "selected_without_validation": asdict(selected)}
(OUTPUT / "split-manifest.json").write_text(json.dumps(manifest, indent=2))
display(cal_summary)
print("Selected from calibration only:", selected.name)
display(val_summary)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
for ax, table, title in zip(axes, [cal_summary, val_summary], ["Calibration: used to select", "Holdout: not used to select"]):
    bars = ax.barh(table.config, table.pq, color=["#83b7a2" if n == selected.name else "#d9c8b5" for n in table.config])
    ax.bar_label(bars, fmt="%.3f", padding=4)
    ax.set(title=title, xlabel="Local panoptic quality (PQ)", xlim=(0, max(.15, float(table.pq.max()) * 1.3)))
plt.show()

### 4. Check a real held-out image
The panels use an official training image. Colored regions are separate predictions.
The final panel shows one annotator's masks. Annotators can disagree.

PQ rewards masks that overlap the right object. It also penalizes missing and extra objects.
Our evaluator follows the host's public reference. A local score is not a leaderboard score.

In [ ]:
example_name = validation[0]
example = read_image(train_paths[example_name])
contrast, disk = image_features(example)
labels = predict_features(contrast, disk, selected)
annotator_id = by_name[example_name][0]
truth = np.zeros_like(labels)
for idx, annotation in enumerate(by_id[annotator_id], 1):
    truth[coco_mask.decode(annotation_rle(annotation)).astype(bool)] = idx
fig, axes = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)
axes[0].imshow(example, cmap="gray", vmin=0, vmax=255)
axes[1].imshow(contrast, cmap="magma", vmin=0, vmax=.2)
for ax, layer in [(axes[2], labels), (axes[3], truth)]:
    ax.imshow(example, cmap="gray", vmin=0, vmax=255)
    ax.imshow(np.ma.masked_where(layer == 0, layer), cmap="turbo", alpha=.8, interpolation="nearest")
for ax, title in zip(axes, ["Official image", "Dark local contrast", "Our predictions", "One annotator's masks"]):
    ax.set_title(title); ax.axis("off")
fig.suptitle(example_name); plt.show()

flow_diagram(["Correct object\nIoU > 0.5", "Extra object\nfalse positive", "Missed object\nfalse negative"], "PQ = matched IoU sum / (TP + 0.5 FP + 0.5 FN)")

### 5. Run all test images
Use image pixels only. Do not read test labels or outside metadata.
Masks must be separate. Each row uses the original image ID and one unique suffix.
The encoded mask always has 2048 × 2048 pixels.

In [ ]:
submission, timing = infer(test_paths, selected, OUTPUT)
print("Saved /kaggle/working/submission.csv")
display(submission.head())
display(timing[["filaments", "seconds"]].describe().round(3))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), constrained_layout=True)
axes[0].hist(timing.filaments, bins=20, color="#e6a16d")
axes[0].set(xlabel="Predicted objects per test image", ylabel="Images", title="Check object counts")
axes[1].hist(timing.seconds, bins=20, color="#83b7a2")
axes[1].set(xlabel="Seconds per image", ylabel="Images", title="Measured CPU time")
plt.show()

### 6. Keep the evidence
The output files record the split, fitted settings, local results, and input hashes.
The checks decode every mask and reject overlap or invalid IDs.

This method can miss faint filaments, lose thin edges, and merge nearby objects.
Next, test better instance separation on the same fixed split.
Then use more months before making a strong performance claim.

In [ ]:
report = {"kind": "local_holdout_and_submission_artifact", "official_score": None,
          "selected_config": asdict(selected), "local_validation": val_summary.to_dict("records"),
          "train_images_available": len(available), "test_images": len(test_paths),
          "submission_rows": len(submission), "runtime_seconds": time.perf_counter() - started,
          "annotation_sha256": hashlib.sha256(annotation_path.read_bytes()).hexdigest(),
          "submission_sha256": hashlib.sha256((OUTPUT / "submission.csv").read_bytes()).hexdigest(),
          "packages": {name: importlib.metadata.version(name) for name in ["numpy", "pandas", "pycocotools", "matplotlib"]}}
(OUTPUT / "run-report.json").write_text(json.dumps(report, indent=2))
display(pd.DataFrame([
    ["submission.csv", "One COCO RLE mask per predicted filament"],
    ["split-manifest.json", "Physical image names and the selected settings"],
    ["validation-detail.csv", "Per-annotator TP, FP, FN, and matched IoU"],
    ["inference-audit.csv", "Input SHA-256, object count, and runtime"],
    ["run-report.json", "Local result, package versions, and output hash"],
    ["submission-checks.json", "Decoded mask checks and zero-object images"],
], columns=["file", "use"]))
print(json.dumps(report, indent=2))

### Sources and reuse
- [Competition and rules](https://www.kaggle.com/competitions/filament-segmentation-2026): data access and submission format.
- [Host self-evaluation notebook](https://www.kaggle.com/code/azimahmadzadeh/self-evaluation-notebook): public PQ matching rules. Our implementation uses COCO mask operations.
- [Panoptic Segmentation](https://arxiv.org/abs/1801.00868): the metric's research source.
- [MAGFiLO data paper](https://doi.org/10.1038/s41597-024-03876-y): annotation provenance.
- [Sijan Regmi's public CPU baseline](https://github.com/regmi-sijan/solar-filament-segmentation-2026): independent precedent for local contrast and connected components. No source code or predictions were copied.

The code here is original and uses the MIT license. Competition images and labels keep their own terms.
The opening art is AI-generated decoration. Scientific plots come from this run.
The prose follows plain-language principles. It is not certified ASD-STE100 text.